# 03 - Gold layer

Business-oriented Gold layer for the Strategist / CTO profile.

## Metric definitions

**Revenue** = sum of line item extended price net of discount:

`revenue = l_extendedprice * (1 - l_discount)`

Tax is not included in the revenue metric.

**Active customer** = a customer with at least one order in the trailing 12 months as of the reporting date.

This notebook builds Gold tables and answers:

1. What is total revenue for 1996-Q1, and how does it compare to the previous quarter?
2. How many active customers were there as of 1998-08-02?

In [0]:
import re

_user = spark.sql("SELECT current_user()").first()[0]
_default_schema = "lakehouse_" + re.sub(r"\W+", "_", _user.split("@")[0]).lower()

dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", _default_schema, "Target schema")

TARGET_CATALOG = dbutils.widgets.get("target_catalog")
TARGET_SCHEMA = dbutils.widgets.get("target_schema")

def silver(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.silver_{table}"

def gold(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.gold_{table}"

print(f"Read from : {TARGET_CATALOG}.{TARGET_SCHEMA}.silver_*")
print(f"Write to  : {TARGET_CATALOG}.{TARGET_SCHEMA}.gold_*")

## Gold tables

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold("daily_revenue")} AS
SELECT
    o.o_orderdate AS revenue_date,
    SUM(l.l_extendedprice * (1 - l.l_discount)) AS revenue
FROM {silver("orders")} o
JOIN {silver("lineitem")} l
    ON o.o_orderkey = l.l_orderkey
GROUP BY o.o_orderdate
ORDER BY o.o_orderdate
""")

display(spark.table(gold("daily_revenue")))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold("customer_orders")} AS
SELECT
    o_orderkey AS order_id,
    o_custkey AS customer_id,
    o_orderdate AS order_date
FROM {silver("orders")}
""")

display(spark.table(gold("customer_orders")))

## Business questions

### Question 1 - Quarterly revenue comparison

In [0]:
q1 = spark.sql(f"""
WITH quarterly_revenue AS (
    SELECT
        DATE_TRUNC('quarter', revenue_date) AS quarter_start,
        SUM(revenue) AS total_revenue
    FROM {gold("daily_revenue")}
    WHERE revenue_date >= DATE '1995-10-01'
      AND revenue_date <  DATE '1996-04-01'
    GROUP BY DATE_TRUNC('quarter', revenue_date)
),
comparison AS (
    SELECT
        quarter_start,
        total_revenue,
        LAG(total_revenue) OVER (ORDER BY quarter_start) AS previous_quarter_revenue
    FROM quarterly_revenue
)
SELECT
    CONCAT(
        YEAR(quarter_start),
        '-Q',
        QUARTER(quarter_start)
    ) AS quarter_label,
    quarter_start,
    total_revenue,
    previous_quarter_revenue,
    total_revenue - previous_quarter_revenue AS absolute_change,
    ROUND(
        (total_revenue - previous_quarter_revenue)
        / previous_quarter_revenue * 100,
        2
    ) AS percentage_change
FROM comparison
ORDER BY quarter_start
""")

display(q1)

Databricks visualization. Run in Databricks to view.

**Answer:** Revenue in 1996-Q1 was approximately **41.23B**, compared with **41.50B** in 1995-Q4. This represents a decrease of approximately **270.14M**, or **0.65% quarter over quarter**.

### Question 2 - Active customers

In [0]:
q2 = spark.sql(f"""
SELECT
    DATE '1998-08-02' AS as_of_date,
    ADD_MONTHS(DATE '1998-08-02', -12) AS window_start_exclusive,
    COUNT(DISTINCT customer_id) AS active_customers
FROM {gold("customer_orders")}
WHERE order_date > ADD_MONTHS(DATE '1998-08-02', -12)
  AND order_date <= DATE '1998-08-02'
""")

display(q2)

Databricks visualization. Run in Databricks to view.

**Answer:** As of **1998-08-02**, there were **433,617 active customers**, where an active customer is defined as a customer with at least one order in the trailing 12 months.

## Validation

Validate that Gold metrics are consistent with the trusted Silver layer.

In [0]:
validation = []

gold_revenue = spark.sql(f"""
SELECT SUM(revenue)
FROM {gold("daily_revenue")}
""").first()[0]

silver_revenue = spark.sql(f"""
SELECT SUM(l_extendedprice * (1 - l_discount))
FROM {silver("lineitem")}
""").first()[0]

validation.append((
    "Revenue reconciles between Silver and Gold",
    gold_revenue == silver_revenue,
    float(gold_revenue),
    float(silver_revenue)
))

gold_orders = spark.table(gold("customer_orders")).count()
silver_orders = spark.table(silver("orders")).count()

validation.append((
    "Order count reconciles between Silver and Gold",
    gold_orders == silver_orders,
    float(gold_orders),
    float(silver_orders)
))

validation_df = spark.createDataFrame(
    validation,
    ["check_name", "passed", "gold_value", "silver_value"]
)

display(validation_df)

assert all(row["passed"] for row in validation_df.collect()), \
    "Gold validation failed"

print("OK: all Gold validation checks passed.")